# lore — VideoMAE + HuBERT with LoRA, 5-fold cross-validation on Colab

Direction `lore` (Paradigm 2): two branches, each a frozen pretrained Transformer adapted with LoRA
(r = 8 on the attention query/value projections), trained separately on the frozen subject-level
folds, then late-fused (weight chosen on `val`). Video = VideoMAE-B on 16 face crops of the speaker;
audio = HuBERT-B on the 16 kHz waveform.

**Before the first run** (once):

1. In your My Drive, a folder (or shortcut) named `dolos` with `dataset.csv`, `splits/`, `audio/`.
2. Upload this direction folder (`lie-detector/notebooks/architectures/lore/`) to My Drive as `lore`. It must
   contain `lore/`, `scripts/`, `tests/` and `inputs/faces-yunet-t32-s224.zip` (the face cache, ~435 MB,
   git-ignored: built by `scripts/build_face_cache.py`).
3. Runtime → Change runtime type → **T4 GPU** (or better).

**Running, in two steps**: the first *Run all* trains **fold 1 only** and stops (section 2 prints its
performance and saves `runs/pilot-fold1/report.txt`). When fold 1 looks right, set `RUN_REMAINING_FOLDS = True`
and *Run all* again for folds 2–5.

**Resuming**: Runtime → Run all. Every fold writes its results to `lore/runs/` on Drive as soon as
it ends; if Colab disconnects, reconnect and *Run all* again — finished folds and the debug run are skipped, and an
interrupted fold resumes from its last finished epoch (`checkpoint.pt`).
Expected cost on a T4: audio ≈ 1 h, video a few hours (the debug cell prints the real time per epoch).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.__version__, torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")

## 0 — Drive, code, dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
MYDRIVE = Path('/content/drive/MyDrive')
DATA = MYDRIVE / 'dolos'                        # shared DOLOS folder (shortcut) or your own upload
# your uploaded direction folder, named `lore` (an older upload named `my_folder` also works)
LORE = next((MYDRIVE / n for n in ('lore', 'my_folder') if (MYDRIVE / n / 'lore' / 'train.py').exists()),
            MYDRIVE / 'lore')
RUNS = LORE / 'runs'                            # results: persistent, on Drive
FACE_CACHE = 'faces-yunet-t32-s224'
OWNER = 'Lorenzo'

for p in (DATA / 'dataset.csv', DATA / 'splits' / 'manifest.json', DATA / 'audio',
          LORE / 'lore' / 'train.py', LORE / 'scripts' / 'run_cv.py', LORE / 'inputs' / f'{FACE_CACHE}.zip'):
    assert p.exists(), f'missing: {p} — see the instructions at the top'
print('Drive OK — code in', LORE)

In [ ]:
# The team repo, only for lie-detector/lib (splits + metrics, never copied)
import os
if not os.path.isdir('/content/DeepLearningCourse/.git'):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
!git -C /content/DeepLearningCourse pull -q
# Pinned: the versions the code was tested with. transformers >= 5.6 is REQUIRED (older VideoMAE
# bypasses LoRA on query/value; the run would fail its LoRA check instead of training silently).
!pip install -q transformers==5.23.0 peft==0.21.3
# Some images ship an old torchao, and peft >= 0.18 raises ImportError when it finds a torchao older
# than 0.16 (even though LoRA never uses it). Nothing here needs torchao: remove it.
!pip uninstall -y -q torchao

In [ ]:
# Face cache -> local disk (read every epoch: never from the Drive mount)
import zipfile, time
FACES_ROOT = Path('/content/faces')
if not (FACES_ROOT / FACE_CACHE / 'cache_config.json').exists():
    t = time.time()
    with zipfile.ZipFile(LORE / 'inputs' / f'{FACE_CACHE}.zip') as z:
        z.extractall(FACES_ROOT)
    print(f'face cache extracted in {time.time() - t:.0f} s')
print(len(list((FACES_ROOT / FACE_CACHE).glob('*.jpg'))), 'face sprites')

os.environ.update({
    'LORE_DATA': str(DATA),
    'LORE_LIB': '/content/DeepLearningCourse/lie-detector',
    'LORE_FACES_ROOT': str(FACES_ROOT),
    'LORE_RUNS': str(RUNS),
    'PYTHONIOENCODING': 'utf-8',
    'HF_HUB_DISABLE_PROGRESS_BARS': '1',
})
# below 20 GB of GPU memory (T4, L4) the video branch recomputes activations instead of storing them
VIDEO_MEMORY_FLAGS = '--grad-checkpointing' if GPU_GB < 20 else ''
%cd {LORE}

In [ ]:
# Sanity checks (CPU, ~1 min): LoRA really in the forward pass; folds never touch the test set.
# A failure STOPS "Run all" here: no GPU time is wasted.
import subprocess

def run_or_stop(cmd, **env):
    proc = subprocess.Popen(cmd, shell=True, env=dict(os.environ, **env), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if 'warn' not in line.lower():
            print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'failed (exit {proc.returncode}): {cmd} -- see the traceback above')

run_or_stop('python tests/test_lora_active.py')
run_or_stop('python tests/test_protocol.py')


## 1 — Debug run (optional, ~10 min)

160/40/40 clips, 2 epochs, fold 1, written to local disk (not to `runs/`). It proves the loop works
on this GPU and prints the seconds per epoch and the peak memory: multiply by ~7 for a full epoch.

In [ ]:
# runs only in the first session: skipped as soon as a real video fold exists on Drive
RUN_DEBUG = not (RUNS / 'video-lora-r8' / 'fold1' / 'metrics.json').exists()
if RUN_DEBUG:
    run_or_stop(f'python scripts/run_cv.py --modality video --debug --folds 1 {VIDEO_MEMORY_FLAGS} '
                '--runs-dir /content/debug_runs')     # stops Run all if it fails
else:
    print('debug run skipped: the video run has already started')

## 2 — Pilot: fold 1 only, then STOP and look at it

Trains **fold 1 only** for both branches, then prints:

* **sanity checks** (`lore/pilot.py`): the fold finished, the train loss went down, the predictions are not
  constant, the best val AUC is at least `MIN_VAL_AUC`; plus the real time of the fold and an estimate for folds 2–5;
* **performance of fold 1**, per branch: val and holdout ACC / F1 / AUC / balanced accuracy / precision /
  recall, the confusion matrix, holdout at the val-tuned threshold and at 0.5, the majority baseline, the
  learning curve and accuracy by gender;
* **fold-1 late fusion** (video + audio, weight tuned on val), written to `runs/pilot-fold1/`.

Everything is also saved to `runs/pilot-fold1/report.txt` on Drive. One fold is ~280 holdout clips: read it
as "does it learn and is it above the baseline", not as the final number.

Sections 3–5 do **nothing** until you set `RUN_REMAINING_FOLDS = True` yourself (next cell).

In [ ]:
MIN_VAL_AUC = 0.5
RUNS_MAIN = ['audio-lora-r8', 'video-lora-r8']
!python scripts/run_cv.py --modality audio --folds 1 2>&1 | grep --line-buffered -v -i warn
!python scripts/run_cv.py --modality video --folds 1 {VIDEO_MEMORY_FLAGS} 2>&1 | grep --line-buffered -v -i warn

import sys
sys.path.insert(0, str(LORE))
from lore.pilot import check_pilot, describe_fold
PILOT_OK, lines = check_pilot(RUNS, RUNS_MAIN, fold=1, min_val_auc=MIN_VAL_AUC)
report = ['=== sanity checks ===', *lines, '', '=== fold 1 performance ===']
for run in RUNS_MAIN:
    report += [describe_fold(RUNS / run, 1), '']
PILOT_DIR = RUNS / 'pilot-fold1'
PILOT_DIR.mkdir(parents=True, exist_ok=True)
(PILOT_DIR / 'report.txt').write_text('\n'.join(report) + '\n', encoding='utf-8')
print('\n'.join(report))

In [ ]:
# fold-1 late fusion (only fold 1, written apart from the final fusion: it is not the board number)
if all((RUNS / r / 'fold1' / 'metrics.json').exists() for r in RUNS_MAIN):
    !python scripts/fuse.py --video-run video-lora-r8 --audio-run audio-lora-r8 --folds 1 --out-dir "{PILOT_DIR}" 2>&1 | grep -v -i warn | sed -n '1,8p'
else:
    print('fold-1 fusion skipped: a branch has no finished fold 1')

### Start folds 2–5 by hand

After reviewing fold 1, set `RUN_REMAINING_FOLDS = True` and run this cell and the ones below (or *Run all*:
fold 1 is already on Drive, so it is skipped).

In [ ]:
RUN_REMAINING_FOLDS = False     # set to True by hand once fold 1 looks right
GO_ON = RUN_REMAINING_FOLDS
if not GO_ON:
    print('folds 2-5 not started: review fold 1 above, then set RUN_REMAINING_FOLDS = True')
elif not PILOT_OK:
    print('WARNING: the fold-1 sanity checks failed (see above); starting folds 2-5 anyway, as requested')

## 3 — Audio branch: HuBERT-B + LoRA, folds 2–5

Writes `runs/audio-lora-r8/fold{2..5}/`. Re-running skips the folds already done.
Runs only with `RUN_REMAINING_FOLDS = True`.

In [ ]:
if GO_ON:
    !python scripts/run_cv.py --modality audio 2>&1 | grep --line-buffered -v -i warn
else:
    print('skipped: RUN_REMAINING_FOLDS = False')

## 4 — Video branch: VideoMAE-B + LoRA on face crops, folds 2–5

The long one. If the session dies, reconnect and run all again: it restarts from the first
unfinished fold. To split it across sessions explicitly, add e.g. `--folds 2-3`, then `--folds 4-5`.

In [ ]:
if GO_ON:
    !python scripts/run_cv.py --modality video {VIDEO_MEMORY_FLAGS} 2>&1 | grep --line-buffered -v -i warn
else:
    print('skipped: RUN_REMAINING_FOLDS = False')

## 5 — Late fusion, report, results-board row

Video-only, audio-only, fused (weight tuned on `val`) and fused (w = 0.5): CV mean ± std,
pooled accuracy, majority baseline, per gender. Writes `runs/fusion-video-lora-r8+audio-lora-r8/`
with `preds/cv_lore.csv` and `board_row.md` — paste the row in `lie-detector/README.md`, `## Results board`.

In [ ]:
if GO_ON:
    !python scripts/fuse.py --video-run video-lora-r8 --audio-run audio-lora-r8 --owner "{OWNER}"
else:
    print('skipped: not all folds were run (RUN_REMAINING_FOLDS = False)')


## 6 — Ablations (optional, for the report)

* **head-only** — same backbones frozen, only the linear head trained: tells whether LoRA adapting
  the backbone is worth it (the central claim of this paradigm). Cheap: no backward through the backbone.
* **LoRA on the MLP of the last 4 blocks too** / **DoRA** — the variations listed in `docs/architectures.md`.

Each writes its own run folder; fuse any pair with `scripts/fuse.py`.

In [ ]:
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    !python scripts/run_cv.py --modality audio --head-only 2>&1 | grep --line-buffered -v -i warn
    !python scripts/run_cv.py --modality video --head-only {VIDEO_MEMORY_FLAGS} 2>&1 | grep --line-buffered -v -i warn
    !python scripts/fuse.py --video-run video-headonly --audio-run audio-headonly --owner "{OWNER}"
    # !python scripts/run_cv.py --modality video --lora-mlp-last-n 4 {VIDEO_MEMORY_FLAGS} 2>&1 | grep --line-buffered -v -i warn
    # !python scripts/run_cv.py --modality video --dora {VIDEO_MEMORY_FLAGS} 2>&1 | grep --line-buffered -v -i warn

## 7 — The paper's official split (optional, reported separately — never on the board)

Same model and settings, trained and tested on the paper's **3 clip-level folds** (`official_fold{1,2,3}.json`),
to compare with the paper's numbers. Run names end in `-official`; they never mix with the subject-level runs.

* The paper's folds have no validation set, so a fixed, stratified **15% of each fold's train** is set aside for
  early stopping and the threshold (same clips for audio and video). The test fold is untouched; the 0–1 clips
  the paper lists on both sides stay in test only.
* Two numbers per branch: **ours** (epoch and threshold chosen on that val, the test fold scored once) and
  **paper-style** (the paper's `train_test.py` keeps the epoch with the best accuracy *on the test fold* at
  threshold 0.5 — optimistic). Only the paper-style one is comparable with the paper's 64.75%.
* Why it is not on the board: the folds are clip-level, so the same people (and episodes) are in train and test,
  including people of the reserved test set. It measures something easier than our protocol.

About the cost of 3 subject-level folds. Results: `runs/fusion-video-lora-r8-official+audio-lora-r8-official/comparison.txt`.

In [ ]:
RUN_OFFICIAL = False    # set to True by hand to run the paper-split comparison
if RUN_OFFICIAL:
    !python scripts/run_cv.py --modality audio --protocol official 2>&1 | grep --line-buffered -v -i warn
    !python scripts/run_cv.py --modality video --protocol official {VIDEO_MEMORY_FLAGS} 2>&1 | grep --line-buffered -v -i warn
    !python scripts/fuse.py --video-run video-lora-r8-official --audio-run audio-lora-r8-official --owner "{OWNER}"